# EGFR human/mouse minibinder design

This notebook directly uses ColabDesign to hallucinate one binder against human and mouse EGFR domain III. It then passes the generated sequence/backbones to ProteinMPNN and downstream validation. BindCraft is not installed or executed; its multi-template notebook is used only as a methodological reference.

Relaxation is intentionally disabled in this stage. pH optimization is deferred.

## 1. Install ColabDesign and lightweight validation dependencies

Run this cell once after selecting a GPU runtime. ColabDesign performs the direct AF2-based binder hallucination; AlphaFold parameter files are required under `PROJECT_ROOT / 'params'`.

In [ ]:
%pip -q install numpy biopython pandas tqdm 'fair-esm[esmfold]' plip
%pip -q install git+https://github.com/sokrypton/ColabDesign.git

# BindCraft is not installed or called by this pipeline. Its multi-template notebook
# was used only as a reference for the direct ColabDesign implementation.

## 2. Import the project helpers

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path('/content/egfr_design')
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)

# The repository is copied into Colab in the next cell.
REPO_ROOT = PROJECT_ROOT / 'week1'
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from egfr_pipeline.target_prep import TARGETS, domain_iii_window, download_targets, trim_pdb_by_residue_range, validate_target_specs
from egfr_pipeline.bindcraft_stage import clear_gpu_memory, gpu_report
from egfr_pipeline.structural_checks import early_structure_report, aligned_binder_rmsd, aligned_complex_rmsd
from egfr_pipeline.colabdesign_stage import HallucinationConfig, generate_shared_binder
from egfr_pipeline.mpnn_stage import PositionConstraint, build_colabdesign_mpnn_options, build_proteinmpnn_jsonl_record, make_histidine_bias, write_constraint_record
from egfr_pipeline.esmfold_stage import clear_esmfold_memory, load_esmfold, screen_complexes_by_target, screen_monomers, write_screening_csv
from egfr_pipeline.plip_stage import parse_plip_targets, run_plip_xml
from egfr_pipeline.colabfold_stage import parse_colabfold_targets
from egfr_pipeline.results import assemble_binder_metric_rows, write_binder_metrics_csv

validate_target_specs()
ESMFOLD_HELPERS = (clear_esmfold_memory, load_esmfold, screen_complexes_by_target, screen_monomers, write_screening_csv)
METRIC_HELPERS = (aligned_binder_rmsd, aligned_complex_rmsd, parse_plip_targets, run_plip_xml, parse_colabfold_targets, assemble_binder_metric_rows, write_binder_metrics_csv)
print({name: target.residue_count for name, target in TARGETS.items()})
print('GPU:', gpu_report())
print('Direct ColabDesign hallucination is available:', generate_shared_binder.__name__)
print('ESMFold staged helpers:', len(ESMFOLD_HELPERS), 'metric helpers:', len(METRIC_HELPERS))

## 3. Define the paired design campaign

Both target batches are used directly by ColabDesign in one shared optimization. Binder lengths are restricted to 65-75 aa for this first campaign.

In [ ]:
CAMPAIGN = {
    'name': 'egfr_human_mouse_stage1',
    'binder_lengths': (65, 75),
    'target_window_start': 345,
    'target_window_length': 150,
    'target_chain': 'A',
    'binder_chain': 'B',
    'final_designs': 30,
    'relaxation': False,
    'ph_optimization': False,
}

for name, target in TARGETS.items():
    start, end, sequence = domain_iii_window(
        target, CAMPAIGN['target_window_start'], CAMPAIGN['target_window_length']
    )
    print(f'{name}: sequence residues {start}-{end}, length={len(sequence)}')

## 4. Download and trim the target structures

The human template is 6ARU chain A. The mouse template is the AlphaFold DB model for Q01279. The residue ranges below use structure author numbering; inspect the printed ranges before launching design because AlphaFold DB and experimental PDB numbering can differ.

In [ ]:
RAW_DIR = PROJECT_ROOT / 'raw_targets'
TRIMMED_DIR = PROJECT_ROOT / 'trimmed_targets'
raw_paths = download_targets(RAW_DIR)

# These are deliberately explicit rather than silently inferred from sequence indices.
# Confirm numbering in the downloaded PDBs before production runs.
PDB_RANGES = {
    'human': (345, 494),
    'mouse': (345, 494),
}

trimmed_paths = {}
for name, target in TARGETS.items():
    start, end = PDB_RANGES[name]
    trimmed_paths[name] = trim_pdb_by_residue_range(
        raw_paths[target.name], TRIMMED_DIR / f'{name}_egfr_domain3.pdb', start, end, CAMPAIGN['target_chain']
    )
    print(name, trimmed_paths[name])

assert all(path.exists() for path in trimmed_paths.values())

## 5. Direct ColabDesign binder hallucination

This is the actual backbone/interface generation stage. It creates one ColabDesign binder model, prepares separate human and mouse target batches, and optimizes one shared binder sequence by aggregating gradients from both targets at every iteration. BindCraft is not executed.

In [ ]:
AF_PARAMS_DIR = PROJECT_ROOT / 'params'
DESIGN_OUTPUT = PROJECT_ROOT / 'outputs' / CAMPAIGN['name'] / 'colabdesign'

hallucination_config = HallucinationConfig(
    binder_length=70,
    iterations_logits=40,
    iterations_soft=30,
    iterations_hard=10,
    num_recycles=1,
    num_models=1,
    sample_models=True,
)

# This call performs direct differentiable binder generation with ColabDesign.
# It requires AlphaFold parameter files under AF_PARAMS_DIR.
generation = generate_shared_binder(
    template_pdbs={
        'human': trimmed_paths['human'],
        'mouse': trimmed_paths['mouse'],
    },
    output_dir=DESIGN_OUTPUT,
    af_params_dir=AF_PARAMS_DIR,
    config=hallucination_config,
    chain=CAMPAIGN['target_chain'],
    seed=17,
    verbose=True,
)

designs = [{
    'design_id': f"{CAMPAIGN['name']}_colabdesign_seed17",
    'sequence': generation['sequence'],
    'source': 'direct_colabdesign_hallucination',
    'colabdesign_human_plddt': generation.get('human_plddt'),
    'colabdesign_mouse_plddt': generation.get('mouse_plddt'),
    'colabdesign_human_iptm': generation.get('human_iptm'),
    'colabdesign_mouse_iptm': generation.get('mouse_iptm'),
}]
print('Generated shared binder:', designs[0]['design_id'], len(designs[0]['sequence']))

## 6. Early structural checks for generated ColabDesign complexes

Use this cell on any direct ColabDesign trajectory or MPNN complex PDB that contains target chain A and binder chain B. It rejects obvious failures without relaxation: no interface contacts or heavy-atom clashes. These checks are triage only and do not establish affinity.

In [ ]:
def check_generated_complexes(complex_dir):
    complex_dir = Path(complex_dir)
    reports = []
    for pdb_path in sorted(complex_dir.glob('*.pdb')):
        reports.append(early_structure_report(
            pdb_path,
            target_chain=CAMPAIGN['target_chain'],
            binder_chain=CAMPAIGN['binder_chain'],
        ))
    return reports

# Example after a smoke test:
# reports = check_generated_complexes(PROJECT_ROOT / 'outputs' / CAMPAIGN['name'] / 'Trajectory')
# for report in reports:
#     print(report)

print('Structural checker ready; no generated complexes have been scanned yet.')

## 7. Stage boundary and memory cleanup

Before moving from hallucination/backbone generation to ProteinMPNN, clear model objects and CUDA allocations. MPNN, ESMFold, and later validation will be added as separate resumable stages.

In [ ]:
clear_gpu_memory()
print('GPU after cleanup:', gpu_report())
print('Next stage: run direct ColabDesign hallucination, then pass its shared sequence and trajectory PDBs to ProteinMPNN.')

## 8. ProteinMPNN redesign and positional controls

ProteinMPNN can steer positions in three distinct ways. `fixed=True` preserves the backbone residue and is the strongest constraint. `allowed_aas='HDE'` restricts sampling to a set. `bias={'H': 1.5}` increases histidine probability without forcing it. Positions are **1-based and chain-local on binder chain B**, not PDB residue numbers and not zero-based Python indexes.

For future pH campaigns, start with soft histidine biasing and compare multiple strengths. This is a sequence-sampling preference, not a pKa or affinity prediction.

In [ ]:
# Example: exploratory pH-oriented redesign. Replace positions after inspecting the interface.
BINDER_LENGTH = 70
constraints = [
    make_histidine_bias(position=12, strength=1.5, label='candidate_pH_site'),
    PositionConstraint(position=28, allowed_aas='DEH', label='acidic_or_histidine_site'),
    # PositionConstraint(position=41, fixed=True, label='preserve_interface_residue'),
]
mpnn_options = build_colabdesign_mpnn_options(BINDER_LENGTH, constraints, omit_aas='C')
print(mpnn_options)
constraint_record = build_proteinmpnn_jsonl_record('A' * BINDER_LENGTH, constraints)
write_constraint_record(constraint_record, PROJECT_ROOT / 'settings' / 'mpnn_constraints_example.json')
print('Saved an auditable constraint record. Apply these controls in the ProteinMPNN wrapper used by the installed ColabDesign version.')

## 9. ESMFold monomer-first validation

Run ESMFold on binder sequences alone first. Only monomers with binder pLDDT >= 75 continue to complex screening. ESMFold pLDDT is a plausibility filter, not an affinity measurement. pTM is not exposed consistently by all fair-esm versions, so the helper records it as unavailable rather than inventing a value.

In [ ]:
# designs must contain design_id and sequence, for example from a FASTA/CSV exported by MPNN.
# designs = [{'design_id': 'd001', 'sequence': '...'}, ...]
# esm_model = load_esmfold(chunk_size=64)
# monomer_pass = screen_monomers(esm_model, designs, PROJECT_ROOT / 'esmfold' / 'monomers', minimum_binder_plddt=75.0)
# clear_esmfold_memory()

# These variables remain empty until the GPU-backed cells above are enabled.
esm_model = globals().get('esm_model')
monomer_pass = globals().get('monomer_pass', [])

# Every monomer-passing binder is repredicted against BOTH trimmed EGFR targets.
# domain_iii_window() supplies the same 150-residue sequence windows used for campaign planning.
if esm_model is not None and monomer_pass:
    trimmed_target_sequences = {
        name: domain_iii_window(target, CAMPAIGN['target_window_start'], CAMPAIGN['target_window_length'])[2]
        for name, target in TARGETS.items()
    }
    complex_results = screen_complexes_by_target(
        esm_model,
        monomer_pass,
        trimmed_target_sequences,
        PROJECT_ROOT / 'esmfold' / 'complexes',
        separator_mode='colon',
    )
    write_screening_csv(complex_results, PROJECT_ROOT / 'esmfold' / 'complex_screening.csv')
    print(f'{len(monomer_pass)} monomers passed; {len(complex_results)} target-specific complexes predicted')
    print('Each design should have one human and one mouse result.')
else:
    print('Define designs, esm_model, and monomer_pass above to run human/mouse complex screening.')
    print("If the installed ESMFold wrapper rejects ':', rerun with separator_mode='glycine'. The fallback is only a heuristic because a G-linker is not a true chain break.")

## 10. PLIP interactions and final metric CSV

Run PLIP externally or in Colab on each initial and ESMFold complex PDB with XML output enabled. The parser counts hydrogen bonds, hydrophobic contacts, salt bridges, pi-stacking, pi-cation, halogen, water-bridge, and metal-complex interactions when the XML explicitly identifies both chain A (target) and chain B (binder). Unassigned XML records are retained as diagnostic counts.

The final CSV has one row per binder. Human and mouse metrics are prefixed separately, and ESMFold binder RMSD is calculated after fitting the ESMFold complex to the corresponding initial ColabFold/BindCraft complex using shared target-chain CA atoms.

In [ ]:
# Provide these maps after the design/validation stages have produced files.
# designs = [{'design_id': 'd001', 'sequence': '...'}, ...]
# complex_results comes from screen_complexes_by_target().
# reference_complexes = {'d001': {'human': '/content/.../initial_human.pdb', 'mouse': '/content/.../initial_mouse.pdb'}}
# plip_xmls = {'d001': {'human': '/content/.../human_report.xml', 'mouse': '/content/.../mouse_report.xml'}}
# colabfold_score_jsons = {'d001': {'human': '/content/.../human_scores.json', 'mouse': '/content/.../mouse_scores.json'}}


def write_final_binder_metrics(
    designs,
    complex_results=None,
    bindcraft_metrics=None,
    structural_metrics=None,
    mpnn_metrics=None,
    monomer_metrics=None,
    reference_complexes=None,
    plip_xmls=None,
    colabfold_score_jsons=None,
):
    colabfold_metrics = {
        design_id: parse_colabfold_targets(score_paths, chain_pair='A-B')
        for design_id, score_paths in (colabfold_score_jsons or {}).items()
    }
    rows = assemble_binder_metric_rows(
        designs=designs,
        bindcraft_metrics=bindcraft_metrics,
        structural_metrics=structural_metrics,
        mpnn_metrics=mpnn_metrics,
        monomer_metrics=monomer_metrics,
        complex_results=complex_results,
        reference_complexes=reference_complexes,
        plip_xmls=plip_xmls,
        colabfold_metrics=colabfold_metrics,
        target_chain='A',
        binder_chain='B',
    )
    output_csv = PROJECT_ROOT / 'outputs' / CAMPAIGN['name'] / 'binder_metrics.csv'
    write_binder_metrics_csv(rows, output_csv)
    print(f'wrote {len(rows)} binder rows to {output_csv}')
    return rows

# final_rows = write_final_binder_metrics(
#     designs, complex_results, bindcraft_metrics, structural_metrics,
#     mpnn_metrics, monomer_metrics, reference_complexes, plip_xmls,
#     colabfold_score_jsons,
# )